# Train the thermal + radar students (v3)

Run the cells in order. Each one prints what it checked, so a wrong dataset or
stale code shows up before an hour of GPU time is spent on it.

**Runtime → Change runtime type → A100** before starting.

## 1 · Drive

In [ ]:
import os
DRIVE = '/content/drive'
REMOTE = DRIVE + '/MyDrive/thermal-fusion/gexport/v3'

# A dead mount leaves files behind and every plain mount() then fails with
# "Mountpoint must not already contain files" - so check reachability first
# and only force a remount when the data is genuinely not there.
if not os.path.exists(REMOTE + '/manifest.json'):
    from google.colab import drive
    try:
        drive.mount(DRIVE)
    except ValueError:
        try:
            drive.flush_and_unmount()
        except Exception:
            pass
        drive.mount(DRIVE, force_remount=True)

assert os.path.exists(REMOTE + '/manifest.json'), (
    'Drive still not reachable. Runtime > Disconnect and delete runtime, '
    'then run this cell again.')
print('drive OK ->', REMOTE)

## 2 · GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU (A100)'
print('GPU:', torch.cuda.get_device_name(0),
      '| %.0f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 3 · Copy the dataset to local disk

Training reads every shard once. Doing that over the Drive mount is what
killed the earlier runs (`Transport endpoint is not connected` mid-read), so
the whole set is copied to the machine's own disk first. The `code/` directory
is always re-copied even when the data is already local, because a stale copy
of it silently trains yesterday's model.

In [ ]:
import subprocess, shutil, time, json, glob
DATA = '/content/v3'

if not os.path.exists(DATA + '/manifest.json'):
    t0 = time.time()
    subprocess.run(['cp', '-r', REMOTE, DATA], check=True)
    print('copied in %.1f min' % ((time.time() - t0) / 60))
else:
    print('data already local')

shutil.rmtree(DATA + '/code', ignore_errors=True)
shutil.copytree(REMOTE + '/code', DATA + '/code')

man = json.load(open(DATA + '/manifest.json'))
print('sessions   :', len(man['sessions']))
print('train      :', man['split']['train'])
print('val        :', man['split']['val'])
print('shards     :', len(glob.glob(DATA + '/*.npz')))
print('loss fix   :', 'negative_weight' in
      open(DATA + '/code/perception/students.py').read())

## 4 · Check the class balance

The trap this dataset was rebuilt to escape: with far more person-frames than
verified-empty ones, the cheapest way to a low loss is to answer "person"
always - which scores a perfect F1 on a val split that has no empty frames,
and paints boxes on bare walls in the field. Training now weights the empty
frames; this cell shows the ratio it will use.

In [ ]:
import numpy as np, collections
train_sessions = set(man['split']['train'])
counts = {'thermal': collections.Counter(), 'radar': collections.Counter()}
for f in glob.glob(DATA + '/*.npz'):
    if f.split('/')[-1].rsplit('-', 1)[0] not in train_sessions:
        continue
    z = np.load(f)
    for plane in counts:
        for v in z[plane + '_label_state']:
            counts[plane][int(v)] += 1
for plane, c in counts.items():
    ratio = c[1] / max(c[0], 1)
    print('%-8s %6d person / %5d verified-empty  -> weight %.1f'
          % (plane, c[1], c[0], min(ratio, 20.0)))
    if c[0] == 0:
        print('   WARNING: no empty frames - this student cannot learn "nobody here"')

## 5 · Train

Output streams live, one line per epoch. Checkpoints are written to **Drive**
after every epoch, so a dropped runtime resumes instead of restarting: just
re-run this cell.

`STUDENT`: `'both'` (~90 min on A100), or `'thermal'` / `'radar'` alone.

In [ ]:
import sys
STUDENT    = 'both'
EPOCHS     = 50
BATCH_SIZE = 32
WORKERS    = 2
OUT        = REMOTE + '/models'

env = dict(os.environ)
env['PYTHONPATH'] = DATA + '/code'
cmd = [sys.executable, '-u', '-m', 'perception.train_students',
       '--data', DATA, '--out', OUT, '--student', STUDENT,
       '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
       '--workers', str(WORKERS)]
print(' '.join(cmd), '\n')

p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print('=== exit code:', p.returncode, '===')
assert p.returncode == 0, 'training failed - the traceback is above'

## 6 · Results

`false_positive_rate` is the number that matters now: it is measured on
held-out sessions that contain no people at all, including a dark empty room.

In [ ]:
import torch, json, os
for name in ('thermal_student.pt', 'radar_student.pt'):
    path = OUT + '/' + name
    if not os.path.exists(path):
        continue
    ck = torch.load(path, map_location='cpu', weights_only=False)
    print('===', name, '| trained', ck.get('created_utc'))
    print('val:', ck.get('val_sessions'))
    print(json.dumps(ck.get('metrics'), indent=1), '\n')

## 7 · Export to ONNX for the Jetson

Writes `*_student.onnx` next to the checkpoints on Drive; the Jetson turns
them into TensorRT engines with `trtexec` and `live.py --students` draws them.

In [ ]:
!pip -q install onnx onnxscript onnxruntime
!cd {DATA} && PYTHONPATH=code python3 -m perception.export_students_onnx --data .
import glob, shutil
for f in glob.glob(DATA + '/models/*.onnx*'):
    shutil.copy(f, OUT)
    print('->', os.path.basename(f))
print(sorted(os.listdir(OUT)))